# The Cusp Catastrophe and Optical Caustics
## Pearcey Beams, Ray Folding, and Wave Interference

This notebook adapts the pseudo-spectral PDE solver to simulate the formation of a **cusp caustic**. In optics and fluid dynamics, a caustic is the envelope of light rays (or wave trajectories) reflected or refracted by a curved surface. At the caustic, the wave amplitude theoretically diverges, creating a bright, sharp curve of focused energy.

---

## 1. The Cusp Catastrophe

In René Thom's and Vladimir Arnold's catastrophe theory, the **cusp catastrophe** is one of the seven elementary catastrophes. It occurs when a 1D system's potential energy has two local minima separated by a maximum, and the control parameters are varied. 

In wave optics, the mathematical description of the wave field near a cusp caustic is given by the **Pearcey integral**. Unlike a simple "fold" caustic (which looks like a bright line), a cusp caustic features a sharp, beak-like point where two fold curves meet, creating a complex internal interference pattern of diffraction fringes.

---

## 2. Physical Setup: The Pearcey Beam

Instead of using a complex spatially varying refractive index (which can introduce numerical instabilities), we can generate a perfect cusp caustic in a *uniform* medium by carefully designing the **initial phase** of the wave.

We initialize a "Pearcey beam": a Gaussian wave packet propagating in the $+x$ direction, but with a **cubic phase modulation** in the transverse $y$ direction.

$$
u(x,y,0) = \exp\left(-\frac{x^2}{2\sigma_x^2} - \frac{y^2}{2\sigma_y^2}\right) \cos(k_x x + \alpha y^3)
$$

The cubic term $\alpha y^3$ acts as a spatially varying "kick" to the transverse momentum of the wave. As the beam propagates, the wavefronts fold over themselves, naturally evolving into the iconic cusp caustic geometry.

---

## 3. The Governing Equation

Because the medium is uniform, the governing equation is the standard 2D scalar wave equation:

$$
\frac{\partial^2 u}{\partial t^2} = c^2 \nabla^2 u
$$

The principal symbol in the pseudo-spectral framework is simply:

$$
a(\xi, \eta) = c^2 (\xi^2 + \eta^2)
$$

All the complex catastrophe geometry arises purely from the initial conditions and the linear superposition of the wave's Fourier components!

# Implementation
## 0. Imports

In [ ]:
from solver import PDESolver, psiOp
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters

In [ ]:
# ── Wave speed ──
C_SQUARED = 1.0        # c² (m²/s²)
C = np.sqrt(C_SQUARED)

# ── Pearcey Beam Parameters ──
K_X = 12.0             # Longitudinal wavenumber (controls carrier frequency)
ALPHA_CUBIC = 0.4      # Strength of the cubic phase modulation (controls the cusp sharpness)

# ── Envelope widths ──
SIGMA_X = 2.5          # Longitudinal envelope width
SIGMA_Y = 3.5          # Transverse envelope width

# ── Dissipation ──
GAMMA = 0.0            # No damping; we want to see the pure interference pattern

# ── Grid and Time ──
# High resolution is required to resolve the fine diffraction fringes inside the caustic
Lx, Ly   = 10.0, 10.0
# Nx, Ny = 32, 32 
# Nx, Ny = 64, 64    
Nx, Ny = 128, 128    
# Nx, Ny = 128, 256    

# Lt, Nt   = 20.0, 400
Lt, Nt   = 30.0, 600
# Lt, Nt   = 40.0, 800
# Lt, Nt   = 50.0, 1000
# Lt, Nt   = 60.0, 1200
n_frames = 300

## 2. Grid setup

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')   # shape (Nx, Ny)

## 3. SymPy symbols and principal symbol

In [ ]:
x, y, t = sp.symbols('x y t', real=True)
xi, eta = sp.symbols('xi eta', real=True)
u_func  = sp.Function('u')
u       = u_func(t, x, y)

# Standard isotropic wave symbol
symbol_wave = C_SQUARED * (xi**2 + eta**2)

print("Principal symbol (Uniform Medium):")
print("  a(ξ, η) =", symbol_wave)

## 4. Wave equation

In [ ]:
#
#   ∂²u/∂t² = -psiOp(a(ξ), u) - GAMMA·∂u/∂t
#
gamma    = sp.Symbol('gamma', positive=True)
equation = sp.Eq(
    sp.diff(u, t, 2),
    -psiOp(symbol_wave, u) - gamma * sp.diff(u, t)
)
equation_num = equation.subs({gamma: GAMMA})

print("Equation:")
print(f"  ∂²u/∂t² = -psiOp({symbol_wave}, u) - {GAMMA}·∂u/∂t")

## 5. Initial conditions (The Pearcey Beam)

In [ ]:
def initial_condition_cusp(xx, yy):
    """
    Pearcey beam: Gaussian envelope with cubic transverse phase.
    """
    # 2D Gaussian envelope
    env = np.exp(-(xx**2) / (2 * SIGMA_X**2) - (yy**2) / (2 * SIGMA_Y**2))
    
    # Phase: longitudinal carrier + cubic transverse modulation
    phase = K_X * xx + ALPHA_CUBIC * yy**3
    
    return env * np.cos(phase)

def initial_velocity_cusp(xx, yy):
    """
    Initial velocity based on WKB approximation for rightward propagation: v = -c * du/dx
    We compute the exact spatial derivative to prevent initial transients.
    """
    env = np.exp(-(xx**2) / (2 * SIGMA_X**2) - (yy**2) / (2 * SIGMA_Y**2))
    phase = K_X * xx + ALPHA_CUBIC * yy**3
    
    # Derivative of the envelope w.r.t x
    d_env_dx = -(xx / SIGMA_X**2) * env
    
    # Product rule: d/dx [env * cos(phase)]
    du_dx = d_env_dx * np.cos(phase) - env * K_X * np.sin(phase)
    
    return -C * du_dx

## 6. Solver setup

In [ ]:
solver = PDESolver(equation_num)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='dirichlet', # Absorb waves at boundaries to prevent wrap-around
    initial_condition=initial_condition_cusp,
    initial_velocity=initial_velocity_cusp,
    n_frames=n_frames,
    plot=True,
)

## 7. Solve

In [ ]:
frames = solver.solve()

## 8. Visualization

In [ ]:
# Raise the animation size limit
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay=None, # Contours are essential here to see the diffraction fringes!
    mode='surface',    
    physical=True      
)

HTML(ani.to_jshtml())

In [ ]:
ani.save('cusp_catastrophe_caustics.mp4', writer='ffmpeg', fps=20, dpi=100)
print("✅ Saved to cusp_catastrophe_caustics.mp4")